# 병합 충돌 해결

모든 저장소는 임시 폴더에 만들며 실제 GitHub 계정이나 원격 저장소를 변경하지 않습니다.


## 준비

공용 원격과 두 작업 폴더를 만들고 출고 기준을 기록합니다.

In [1]:
# 셀이 만든 저장소만 사용하는 임시 작업 폴더를 준비합니다.
import subprocess
import sys
import tempfile
from pathlib import Path
workspace = tempfile.TemporaryDirectory(prefix="git-lesson-")
root = Path(workspace.name)
# cwd는 명령을 실행할 폴더이며 check=True는 명령 실패를 예외로 알립니다.
def git(*args: str, cwd: Path | None = None, check: bool = True) -> str:
    """지정한 임시 저장소에서 Git 명령을 실행합니다."""
    result = subprocess.run(["git", *args], cwd=cwd or root,
                            text=True, capture_output=True, check=check)
    return result.stdout.strip() or result.stderr.strip()
def configure(repo: Path) -> None:
    """실습 저장소에만 작성자 정보를 설정합니다."""
    git("config", "user.name", "Git Learner", cwd=repo)
    git("config", "user.email", "learner@example.invalid", cwd=repo)

In [2]:
# 두 작업자가 함께 사용할 원격을 준비합니다.
remote = root / "team.git"
git("init", "--bare", "--initial-branch=main", remote.as_posix())
alice, bob = root / "alice", root / "bob"
git("clone", remote.as_posix(), alice.as_posix())
configure(alice)
(alice / "policy.txt").write_text(
    "14시 이전 결제: 오늘 출고\n14시 이후 결제: 내일 출고\n", encoding="utf-8"
)
(alice / "message.txt").write_text("배송 안내: 준비 중\n", encoding="utf-8")
git("add", "policy.txt", "message.txt", cwd=alice)
git("commit", "-m", "docs: 출고 기준 기록", cwd=alice)
git("push", "-u", "origin", "main", cwd=alice)
print("공용 원격 준비 완료")

공용 원격 준비 완료


## 기본 실행

Alice가 정책을 반영한 안내를 먼저 원격에 게시합니다.

In [3]:
# Alice가 수정한 같은 줄을 먼저 원격 main에 반영합니다.
(alice / "message.txt").write_text("배송 안내: 14시 이전은 오늘 출고\n", encoding="utf-8")
git("add", "message.txt", cwd=alice)
git("commit", "-m", "docs: 당일 출고 안내", cwd=alice)
git("push", cwd=alice)
print(git("log", "--oneline", "-2", cwd=alice))

ad4a477 docs: 당일 출고 안내
bf9d19a docs: 출고 기준 기록


## 값 하나를 바꿔 비교

Bob은 이전 main에서 같은 줄을 다른 문장으로 수정합니다.

In [4]:
# Bob의 임시 복제본만 이전 커밋으로 옮겨 서로 다른 변경을 만듭니다.
git("clone", remote.as_posix(), bob.as_posix())
configure(bob)
git("reset", "--hard", "HEAD~1", cwd=bob)
(bob / "message.txt").write_text("배송 안내: 내일 출고\n", encoding="utf-8")
git("add", "message.txt", cwd=bob)
git("commit", "-m", "docs: 익일 출고 안내", cwd=bob)
push = subprocess.run(["git", "push", "origin", "main"], cwd=bob,
                      text=True, capture_output=True)
print("push 종료 코드:", push.returncode)
print("원격보다 이전 이력:", "non-fast-forward" in push.stderr)
assert push.returncode != 0 and "non-fast-forward" in push.stderr

push 종료 코드: 1
원격보다 이전 이력: True


## 실패 재현

Bob이 원격 변경을 가져와 병합하면 같은 줄의 차이에서 충돌합니다.

In [5]:
# --no-rebase로 원격 main을 병합하면 같은 줄의 충돌 표시가 남습니다.
merge = subprocess.run(
    ["git", "pull", "--no-rebase", "origin", "main"], cwd=bob,
    text=True, capture_output=True)
print("종료 코드:", merge.returncode)
print(git("status", "--short", cwd=bob))
print((bob / "message.txt").read_text(encoding="utf-8"))
assert merge.returncode != 0
assert "<<<<<<<" in (bob / "message.txt").read_text(encoding="utf-8")

종료 코드: 1
UU message.txt
<<<<<<< HEAD
배송 안내: 내일 출고
배송 안내: 14시 이전은 오늘 출고
>>>>>>> ad4a4771848f17efd489ac2b98823f4588c603c6



## 수정

저장소의 정책 문서를 근거로 안내를 확정하고 원격에 게시합니다.

In [6]:
# 정책에 맞춘 실제 파일에서 충돌 표시 제거와 병합 완료를 확인합니다.
resolved = "배송 안내: 14시 이전 결제는 오늘 출고, 14시 이후 결제는 내일 출고\n"
(bob / "message.txt").write_text(resolved, encoding="utf-8")
git("add", "message.txt", cwd=bob)
git("commit", "-m", "merge: 출고 정책에 맞춰 충돌 해결", cwd=bob)
git("push", "origin", "main", cwd=bob)
print((bob / "policy.txt").read_text(encoding="utf-8").strip())
print((bob / "message.txt").read_text(encoding="utf-8").strip())
parents = git("show", "--pretty=%P", "--no-patch", "HEAD", cwd=bob)
print("병합 부모:", parents)
saved = (bob / "message.txt").read_text(encoding="utf-8")
assert all(marker not in saved for marker in ("<<<<<<<", "=======", ">>>>>>>"))
assert not git("status", "--porcelain", cwd=bob)
assert len(parents.split()) == 2
assert git("rev-parse", "HEAD", cwd=bob) == git("rev-parse", "origin/main", cwd=bob)

14시 이전 결제: 오늘 출고
14시 이후 결제: 내일 출고
배송 안내: 14시 이전 결제는 오늘 출고, 14시 이후 결제는 내일 출고
병합 부모: 032b1cb833e8eeb143512fb626cb842266952e05 ad4a4771848f17efd489ac2b98823f4588c603c6
